In [ ]:
# Dashboard
# Dashboard.ipynb — Andre Miranda

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import dash_leaflet as dl
import plotly.express as px
import dash

import base64
import pandas as pd
import logging

from CRUD_Python_Module import AnimalShelter
from rescue_scoring import rank_animals, normalize_breeds
from login_page import get_credentials

# ---------------------------------------------------------
# LOGGING
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] Dashboard: %(message)s"
)
logger = logging.getLogger("Dashboard")

JupyterDash.infer_jupyter_proxy_config()

# ---------------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------------
username, password = get_credentials()
shelter = AnimalShelter(username, password)

# ---------------------------------------------------------
# INITIAL DATA
# ---------------------------------------------------------
full_df = pd.DataFrame.from_records(shelter.read_all())
if "_id" in full_df.columns:
    full_df.drop(columns=["_id"], inplace=True)

df = full_df.head(300).copy()

# ---------------------------------------------------------
# LOGO
# ---------------------------------------------------------
def encode_logo(path):
    try:
        return base64.b64encode(open(path, "rb").read()).decode()
    except Exception:
        return ""

encoded_logo = encode_logo("Grazioso Salvare Logo.png")

# ---------------------------------------------------------
# APP + CUSTOM HTML
# ---------------------------------------------------------
app = JupyterDash("GraziosoDashboard")

app.index_string = '''
<!DOCTYPE html>
<html>
<head>
    {%metas%}
    <title>Grazioso Salvare Dashboard</title>
    {%favicon%}
    {%css%}
    <style>
        body {
            background-color: #f5f5f7;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            margin: 0;
            padding: 0;
            color: #1d1d1f;
        }
        .dashboard-container {
            max-width: 1200px;
            margin: 40px auto;
            padding: 30px;
            background-color: #ffffff;
            border-radius: 20px;
            box-shadow: 0 8px 20px rgba(0,0,0,0.08);
        }
        .filter-box {
            display: flex;
            justify-content: center;
            align-items: center;
            text-align: center;
            margin-bottom: 20px;
        }

        #map-card {
            margin-top: 30px;
            padding: 20px;
            background-color: #fafafa;
            border-radius: 20px;
            border: 1px solid #e5e5ea;
            box-shadow: 0 4px 12px rgba(0,0,0,0.05);
            position: relative;
            z-index: 1;
        }

        #info-card {
            position: absolute;
            top: 20px;
            right: 20px;
            width: 240px;
            background-color: rgba(255,255,255,0.85);
            padding: 14px 16px;
            border-radius: 16px;
            box-shadow: 0 6px 18px rgba(0,0,0,0.15);
            backdrop-filter: blur(10px);
            font-size: 14px;
            line-height: 1.5;
            color: #1d1d1f;
            z-index: 999999 !important;
            pointer-events: auto;
        }

        .leaflet-container {
            z-index: 0 !important;
        }

        .aggregation-card {
            margin-top: 30px;
            padding: 24px;
            background-color: #fafafa;
            border-radius: 20px;
            border: 1px solid #e5e5ea;
            box-shadow: 0 4px 12px rgba(0,0,0,0.05);
        }

        .kpi-box {
            flex: 1;
            background-color: #ffffff;
            padding: 20px;
            border-radius: 16px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
            text-align: center;
        }
        .kpi-value {
            font-size: 28px;
            font-weight: 700;
            color: #1d1d1f;
            margin-top: 6px;
        }
        .kpi-label {
            font-size: 14px;
            color: #6e6e73;
        }

        .collapsible-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            cursor: pointer;
            margin-bottom: 10px;
        }
        .collapsible-header h3 {
            margin: 0;
        }
        .collapsible-toggle {
            font-size: 13px;
            color: #007aff;
        }

        .analytics-controls {
            display: flex;
            justify-content: flex-start;
            align-items: center;
            gap: 20px;
            margin-bottom: 20px;
        }
    </style>
</head>

<body>
    <div class="dashboard-container">
        {%app_entry%}
    </div>

    <footer>
        {%config%}
        {%scripts%}
        {%renderer%}
    </footer>
</body>
</html>
'''

# ---------------------------------------------------------
# LAYOUT
# ---------------------------------------------------------
app.layout = html.Div([

    html.Center([
        html.Img(src=f"data:image/png;base64,{encoded_logo}", style={"height": "90px"}),
        html.H1("Grazioso Salvare Dashboard"),
        html.Div("Created by Andre Miranda")
    ]),

    html.Hr(),

    # Rescue filter
    html.Div(
        className="filter-box",
        children=[
            dcc.RadioItems(
                id="rescue-filter",
                options=[
                    {"label": "Water Rescue", "value": "water"},
                    {"label": "Mountain/Wilderness", "value": "mountain"},
                    {"label": "Disaster/Tracking", "value": "disaster"},
                    {"label": "Reset", "value": "reset"}
                ],
                value="water",
                inline=True,
                style={"fontSize": "16px"}
            )
        ]
    ),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i} for i in df.columns] +
                [{"name": "Rescue Score", "id": "rescue_score"}],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"height": "400px", "overflowY": "auto"},
        style_cell={"textAlign": "left", "padding": "6px 8px"},
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", style={"flex": "1"}),
            html.Div(
                id="map-card",
                style={"flex": "1"},
                children=[
                    html.Div(id="info-card"),
                    html.Div(id="map-id")
                ]
            )
        ]
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        id="aggregation-card",
        className="aggregation-card",
        children=[
            html.Div(
                className="collapsible-header",
                children=[
                    html.H3("Database Analytics"),
                    html.Span("Collapse / Expand", id="analytics-toggle", className="collapsible-toggle")
                ]
            ),

            html.Div(
                className="analytics-controls",
                children=[
                    dcc.RadioItems(
                        id="analytics-mode",
                        options=[
                            {"label": "Filtered Analytics", "value": "filtered"},
                        ],
                        value="filtered",
                        inline=True,
                        style={"fontSize": "16px"}
                    )
                ]
            ),

            html.Div(id="aggregation-output")
        ]
    )
])

# ---------------------------------------------------------
# TABLE + SCORING
# ---------------------------------------------------------
@app.callback(
    Output("datatable-id", "data"),
    Input("rescue-filter", "value"),
    prevent_initial_call=False
)
def update_table(filter_value):
    if filter_value == "water":
        records = shelter.filter_water_rescue()
    elif filter_value == "mountain":
        records = shelter.filter_mountain_rescue()
    elif filter_value == "disaster":
        records = shelter.filter_disaster_rescue()
    elif filter_value == "reset":
        records = full_df.to_dict("records")
    else:
        return df.to_dict("records")

    ranked = rank_animals(records)
    dff = pd.DataFrame.from_records(ranked)

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    if len(dff) > 1000:
        dff = dff.head(1000)

    return dff.to_dict("records")

# ---------------------------------------------------------
# PIE CHART
# ---------------------------------------------------------
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    prevent_initial_call=True
)
def update_graph(viewData):
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" not in dff.columns or dff.empty:
        return []

    all_breeds = []
    for breed_string in dff["breed"]:
        all_breeds.extend(normalize_breeds(breed_string))

    if not all_breeds:
        return []

    breed_counts = pd.Series(all_breeds).value_counts()

    top_n = 8
    top_breeds = breed_counts.head(top_n)

    labels = list(top_breeds.index)
    values = list(top_breeds.values)

    other_total = breed_counts.iloc[top_n:].sum()
    if other_total > 0:
        labels.append("Other Breeds")
        values.append(other_total)

    fig = px.pie(names=labels, values=values, title="Breed Distribution (Normalized)")

    return [dcc.Graph(figure=fig)]

# ---------------------------------------------------------
# MAP + INFO CARD
# ---------------------------------------------------------
@app.callback(
    [Output("map-id", "children"),
     Output("info-card", "children")],
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ],
    prevent_initial_call=False
)
def update_map(viewData, index):
    if not viewData:
        return [], html.Div("No data available.")

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0] if index else 0

    lat = dff.iloc[row].get("location_lat")
    lon = dff.iloc[row].get("location_long")
    rescue_score = dff.iloc[row].get("rescue_score", "N/A")

    info = html.Div([
        html.B("ID: "), str(dff.iloc[row].get("animal_id")), html.Br(),
        html.B("Name: "), str(dff.iloc[row].get("name")), html.Br(),
        html.B("Type: "), str(dff.iloc[row].get("animal_type")), html.Br(),
        html.B("Breed: "), str(dff.iloc[row].get("breed")), html.Br(),
        html.B("Sex: "), str(dff.iloc[row].get("sex_upon_outcome")), html.Br(),
        html.B("Outcome: "), str(dff.iloc[row].get("outcome_type")), html.Br(),
        html.B("Age (weeks): "), str(dff.iloc[row].get("age_upon_outcome_in_weeks")), html.Br(),
        html.B("Rescue Score: "), str(rescue_score)
    ])

    if lat is None or lon is None or pd.isna(lat) or pd.isna(lon):
        fallback_map = [
            dl.Map(
                style={"width": "100%", "height": "500px"},
                center=[30.2672, -97.7431],
                zoom=10,
                children=[dl.TileLayer()]
            )
        ]
        return fallback_map, info

    map_component = [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(dff.iloc[row].get("breed"))),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(str(dff.iloc[row].get("name")))
                        ])
                    ]
                )
            ]
        )
    ]

    return map_component, info

# ---------------------------------------------------------
# COLLAPSIBLE ANALYTICS PANEL
# ---------------------------------------------------------
@app.callback(
    Output("aggregation-output", "style"),
    Input("analytics-toggle", "n_clicks"),
    State("aggregation-output", "style"),
    prevent_initial_call=True
)
def toggle_analytics_panel(n_clicks, current_style):
    if current_style is None:
        current_style = {"display": "block"}

    if current_style.get("display", "block") == "block":
        return {"display": "none"}
    else:
        return {"display": "block"}

# ---------------------------------------------------------
# AGGREGATION SUMMARY
# ---------------------------------------------------------
@app.callback(
    Output("aggregation-output", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("rescue-filter", "value"),
        Input("analytics-mode", "value")
    ],
    prevent_initial_call=True
)
def update_aggregation_summary(viewData, filter_value, analytics_mode):

    # Always filtered analytics
    if filter_value == "reset":
        dff = full_df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData) if viewData else pd.DataFrame()

    if dff.empty:
        return "Aggregation summary unavailable."

    # ⭐ FIX: Coerce age column to numeric
    if "age_upon_outcome_in_weeks" in dff.columns:
        dff["age_upon_outcome_in_weeks"] = pd.to_numeric(
            dff["age_upon_outcome_in_weeks"], errors="coerce"
        )

    # KPI Tiles
    total_animals = len(dff)
    avg_age = dff["age_upon_outcome_in_weeks"].mean() if "age_upon_outcome_in_weeks" in dff.columns else 0
    top_breed = dff["breed"].value_counts().idxmax() if "breed" in dff.columns else "Unknown"
    top_outcome = dff["outcome_type"].value_counts().idxmax() if "outcome_type" in dff.columns else "Unknown"

    kpi_section = html.Div(
        style={"display": "flex", "gap": "20px", "marginBottom": "30px"},
        children=[
            html.Div([
                html.Div("Total Animals (filtered)", className="kpi-label"),
                html.Div(str(total_animals), className="kpi-value")
            ], className="kpi-box"),

            html.Div([
                html.Div("Avg Age (weeks)", className="kpi-label"),
                html.Div(f"{avg_age:.2f}", className="kpi-value")
            ], className="kpi-box"),

            html.Div([
                html.Div("Top Breed (filtered)", className="kpi-label"),
                html.Div(top_breed, className="kpi-value")
            ], className="kpi-box"),

            html.Div([
                html.Div("Most Common Outcome (filtered)", className="kpi-label"),
                html.Div(top_outcome, className="kpi-value")
            ], className="kpi-box"),
        ]
    )

    # Bar Chart (Breed vs Outcome)
    if "breed" in dff.columns and "outcome_type" in dff.columns:
        bar_df = dff.groupby(["breed", "outcome_type"]).size().reset_index(name="count")
        bar_fig = px.bar(
            bar_df,
            x="breed",
            y="count",
            color="outcome_type",
            title="Breed vs Outcome (Filtered)",
            labels={"breed": "Breed", "count": "Count", "outcome_type": "Outcome"},
            barmode="group"
        )
        bar_chart = html.Div(
            style={"marginBottom": "30px"},
            children=[dcc.Graph(figure=bar_fig)]
        )
    else:
        bar_chart = html.Div("Bar chart unavailable for current mode.")

    # Age Distribution Histogram
    if "age_upon_outcome_in_weeks" in dff.columns:
        age_df = dff.dropna(subset=["age_upon_outcome_in_weeks"]).copy()
        if not age_df.empty:
            hist_fig = px.histogram(
                age_df,
                x="age_upon_outcome_in_weeks",
                nbins=30,
                title="Age Distribution Histogram (Filtered)",
                labels={"age_upon_outcome_in_weeks": "Age (weeks)"},
                color_discrete_sequence=["#4A90E2"]
            )
            age_hist_chart = html.Div(
                style={"marginBottom": "30px"},
                children=[dcc.Graph(figure=hist_fig)]
            )
        else:
            age_hist_chart = html.Div("Age distribution unavailable for current mode.")
    else:
        age_hist_chart = html.Div("Age distribution unavailable for current mode.")

    return html.Div([
        kpi_section,
        html.H4("Breed vs Outcome Bar Chart", style={"marginTop": "20px"}),
        bar_chart,
        html.H4("Age Distribution Histogram", style={"marginTop": "20px"}),
        age_hist_chart
    ])

# ---------------------------------------------------------
# RUN APP
# ---------------------------------------------------------
app.run_server(mode="external", port=8051)


SyntaxError: invalid syntax (2601986463.py, line 7)